## EX: Building a RAG Pipeline

In this exercise, we will build a fully functional RAG pipeline using chromadb (a real-world vector database) and the Gemini API. We will:

* **Embed:** Use sentence_transformers to convert text into mathematical vectors locally.

* **Store & Retrieve:** Build a Chroma vector database to ingest text chunks, store their embeddings, and perform a semantic search based on a user's query.

* **Generate:** Send the user's query and the retrieved database context to the Gemini API to generate a grounded answer.

Note: You must install the required packages (pip install chromadb sentence_transformers requests) before running this code.

In [ ]:
# Only run this cell after downloading and selecting your kernel
!python.exe -m pip install --upgrade pip
!pip install sentence_transformers chromadb

In [ ]:
import requests
import json
from sentence_transformers import SentenceTransformer
import chromadb

# Set your secure Gemini API key here
API_KEY = "YOUR_GOOGLE_API_KEY"
GEMINI_API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent?key={API_KEY}"


# Step 1: Procure a method to vectorize text locally
print("Loading Embedding Model...")
embed_model = SentenceTransformer("all-mpnet-base-v2")

def embed(texts):
    """Converts a list of text strings into vector embeddings."""
    return embed_model.encode(texts).tolist()

# Step 2: Create a ChromaDB persistent client and functions
client = chromadb.PersistentClient(path="./my_chroma_db")

def add_documents(client, documents: list, metadatas:list, ids:list, mode="add"):
    """Adds text chunks and their embeddings to the vector database."""
    collection = client.get_or_create_collection(
        name="my_knowledge_base",
        metadata={"hnsw:space": "cosine"},
    )
    
    # Check to make sure docs, metadata, and ids are the same length
    if len(documents) == len(metadatas) == len(ids): 
        # Generate the vector embeddings of the documents
        vectors = embed(documents)

        if mode == "add":
            collection.add(
                embeddings=vectors,
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
        elif mode == "update":
            collection.upsert(
                embeddings=vectors,
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )

def build_chroma_db(client):
    """Simulates loading, chunking, and embedding a dataset into the DB."""
    # Prepare each document by extracting text, adding metadata tags, and naming each document.
    documents = [
        "TCP is a connection-oriented protocol used in networking. This method of communication is more reliable because the connection is initialized using a handshake.",
        "UDP is a connectionless protocol often used for streaming.",
        "NAT allows multiple devices to share one public IP address."
    ]

    metadatas = [
        {"topic": "tcp", "source":"Ch. 1.2"},
        {"topic": "udp", "source":"Ch. 1.2"},
        {"topic": "nat", "source":"Ch. 1.3"}
    ]

    ids = ["doc1", "doc2", "doc3"]
    add_documents(client, documents, metadatas, ids, mode="update")

# Build the vector database (Run this once)
print("Building Vector Database...")
build_chroma_db(client)

# Step 3: RAG helper function to retrieve context
def retrieve_context(client, user_query, k=2):
    """Searches the database for the 'k' most relevant chunks."""
    collection = client.get_or_create_collection(
        name="my_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )
    
    query_emb = embed([user_query])
    # Perform semantic search using Cosine Similarity
    result = collection.query(query_embeddings=query_emb, n_results=k)
    
    documents = result["documents"][0]
    metadatas = result["metadatas"][0]
    ids = result["ids"][0]
    
    output = ""
    for doc, meta, doc_id in zip(documents, metadatas, ids):
        output += f"{doc} Source: {meta.get('source')} (from {doc_id})\n"
    return output

# Step 4: Agent prompt wrapper
def build_rag_prompt(client, user_query):
    """Retrieves context and formats the strict system instruction."""
    context = retrieve_context(client, user_query)
    prompt_for_LLM =f"""You are an assistant using retrieved knowledge. Answer the user's question using ONLY the provided context. Provide an example when possible. If the context does not contain the answer, say "Insufficient knowledge."

    [RETRIEVED CONTEXT]
    {context}

    [USER QUESTION]
    {user_query}
    """
    print(f"\nPROMPT SENT TO LLM: \n {prompt_for_LLM}")
    return prompt_for_LLM

# Step 5: Connect to the Gemini API workflow
def ask_agent(client, user_query):
    """Sends the augmented prompt to Gemini and parses the response."""
    prompt = build_rag_prompt(client, user_query)
    print("\n--- Sending Prompt to Gemini ---")
    
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    headers = {'Content-Type': 'application/json'}
    
    response = requests.post(GEMINI_API_URL, headers=headers, data=json.dumps(payload))
    
    if response.status_code == 200:
        return response.json()['candidates'][0]['content']['parts'][0]['text']
    else:
        return f"Error: {response.status_code}\n{response.text}"

# 6. Execute the query. 
# NOTE: If you change this to a topic not in the vector database, the LLM should respond "Insufficient knowlege"
user_query = "What are some common protocols?"
print(f"\nUser Query: {user_query}")
final_answer = ask_agent(client, user_query)
print(f"\n[AI RESPONSE]: {final_answer}")


Loading Embedding Model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12943.78it/s]


Building Vector Database...

User Query: What are some common protocols?

PROMPT SENT TO LLM: 
 You are an assistant using retrieved knowledge. Answer the user's question using ONLY the provided context. Provide an example when possible. If the context does not contain the answer, say "Insufficient knowledge."

    [RETRIEVED CONTEXT]
    TCP is a connection-oriented protocol used in networking. This method of communication is more reliable because the connection is initialized using a handshake. Source: Ch. 1.2 (from doc1)
UDP is a connectionless protocol often used for streaming. Source: Ch. 1.2 (from doc2)


    [USER QUESTION]
    What are some common protocols?
    

--- Sending Prompt to Gemini ---

[AI RESPONSE]: Common protocols include TCP and UDP. TCP is a connection-oriented, reliable protocol used in networking that initializes connections using a handshake (Source: Ch. 1.2 from doc1). In contrast, UDP is a connectionless protocol often used for streaming (Source: Ch. 1.2 f


### Interpreting the Results

When you run this script, chromadb successfully translates the user query ("What are some common protocols?") into a vector, calculates the cosine similarity against the database, and retrieves doc1 and doc2. The Gemini API is then fed this specific snippet and forced to generate its answer using only the provided, cited information.

